# Member 2 — Preprocessing, Features & Leakage Prevention
**IT3091 Machine Learning — TRIAGE-Telco / Group 2026-DS-07**  
**Responsible member:** Imani H.A.I. (IT24102383)  
**Scope:** Validated inputs; fold-aware encoding/scaling; structural-zero handling; imbalance comparisons; feature experiments; sensitivity handoff

**What is executed afresh:** full preprocessing definitions and a held-out-fold training check, plus demonstration of engineered features. **What is loaded:** the original recorded four-treatment, balancing, feature-ablation, demographic-reduction and sensitivity results. The recorded full experiments are not falsely presented as rerun hyperparameter or preprocessing searches.

**How to run:** Extract the entire four-folder ZIP, install `final_notebook_code/requirements.txt`, and open this notebook from the project folder. Use **Kernel → Restart & Run All**. It reads the original CSV and archived evidence from the folder tree; it does **not** need any other notebook to have run first.

**Evidence boundary:** The historical full-data EDA is descriptive; recorded test results are a historical replay with disclosed prior test exposure; reused development folds make tuning scores potentially optimistic. Notebook sections label archived experimental evidence separately from computations performed when you run the notebook.


In [1]:
from pathlib import Path
import hashlib, warnings, time, sys, platform
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone, BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score, f1_score,
    brier_score_loss, confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve)
from scipy.stats import chi2_contingency
from catboost import CatBoostClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)
print('Python:',platform.python_version(),'| sklearn:',__import__('sklearn').__version__,'| CatBoost:',__import__('catboost').__version__)
print('No local Python scripts are imported; CSVs load from this submission.')

Python: 3.13.5 | sklearn: 1.8.0 | CatBoost: 1.2.8
No local Python scripts are imported; CSVs load from this submission.


In [2]:
# Load the original project files directly. Nothing is compressed or hidden in this notebook.
# Open the notebook from the submission root OR from final_notebook_code/.
from pathlib import Path

def locate_project_root():
    current = Path.cwd().resolve()
    candidates = [current, current.parent, current.parent.parent]
    for base in candidates:
        if (base / 'dataset' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv').exists() and (base / 'model_method_comparison').is_dir():
            return base
    raise FileNotFoundError(
        'Project files not found. Extract the entire ZIP first, then run Jupyter from '
        'the submission folder or final_notebook_code folder. Do not open only the .ipynb.'
    )

PROJECT_ROOT = locate_project_root()
FILES = {
    'raw_csv': 'dataset/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv',
    'split': 'dataset/splits/split_membership.csv',
    'folds': 'dataset/splits/cv_fold_membership.csv',
    'dictionary': 'dataset/DATA_DICTIONARY.csv',
    'eda_log': 'decision_logs/member1/EDA_INSIGHT_LOG.csv',
    'prep_log': 'decision_logs/member2/preprocessing_decision_log.csv',
    'feature_log': 'decision_logs/member2/feature_registry.csv',
    'decision_log': 'decision_logs/project/CONSOLIDATED_DECISION_LOG.csv',
    'models_summary': 'model_method_comparison/MODEL_METHOD_COMPARISON.csv',
    'preprocess_pred': 'model_method_comparison/handoffs/member2_to_member3/OOF_preprocessing_predictions.csv',
    'oof': 'model_method_comparison/handoffs/member3_to_member4/candidate_models_updated_oof.csv',
    'stability': 'model_method_comparison/handoffs/member3_to_member4/customer_stability_integrated_development.csv',
    'generalization': 'model_method_comparison/results/member3/generalization_audit/model_generalization_summary.csv',
    'tolerance': 'model_method_comparison/results/member3/tables/near_optimal_tolerance_comparison.csv',
    'recorded_test': 'model_method_comparison/results/member4/final_test_probabilities_20261002_100144.csv',
    'scorecard': 'model_method_comparison/results/member4/final_test_scorecard.csv',
    'synthetic': 'dataset/examples/synthetic_customers.csv',
    'tc_treatment': 'model_method_comparison/results/member2/tables/totalcharges_treatment_results.csv',
    'balance': 'model_method_comparison/results/member2/tables/imbalance_arena_results.csv',
    'features': 'model_method_comparison/results/member2/tables/feature_ablation_results.csv',
    'demographic': 'model_method_comparison/results/member2/tables/data_minimization_results.csv',
}

def source_csv(name):
    '''Read one of the original recorded CSVs from the four submission folders.'''
    return pd.read_csv(PROJECT_ROOT / FILES[name])

raw = source_csv('raw_csv')
split = source_csv('split')
folds = source_csv('folds')
assert raw.shape == (7043, 21) and raw.customerID.is_unique
assert len(split) == len(raw) and len(folds) == 5634
print('Project folder:', PROJECT_ROOT)
print('Original Telco CSV:', FILES['raw_csv'])
print('Dataset size:', raw.shape, '| Development folds:', len(folds))
display(raw.head(3))


Project folder: /mnt/data/TRIAGE_Telco_FINAL_FOUR_MEMBER_NOTEBOOKS
Original Telco CSV: dataset/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
Dataset size: (7043, 21) | Development folds: 5634


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 2 | Member 2 — Imani: preprocessing, feature engineering and leakage prevention
**Responsibility:** Convert raw inputs into valid model features without learning preprocessing statistics from validation customers. The complete modelling functions are **inside this notebook** — no `pipeline_contract.py` import.

Important preserved rules: `customerID` and `Churn` never enter the predictor matrix; the 11 empty `TotalCharges` are legitimate only at `tenure=0`; no structural service category is treated as ordinary missingness; encoders/scalers are fitted within each training fold.

In [3]:
RAW_FEATURES=['gender','SeniorCitizen','Partner','Dependents','tenure','PhoneService','MultipleLines','InternetService',
    'OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract',
    'PaperlessBilling','PaymentMethod','MonthlyCharges','TotalCharges']
NUMERIC=['SeniorCitizen','tenure','MonthlyCharges','TotalCharges']
CATEGORICAL=[x for x in RAW_FEATURES if x not in NUMERIC]

def semantic_repair(X):
    if not isinstance(X,pd.DataFrame):raise TypeError('Require DataFrame')
    if list(X.columns)!=RAW_FEATURES: raise ValueError('Expected exactly 19 allowed predictors, in registered order')
    if X.columns.duplicated().any():raise ValueError('Duplicated feature')
    data=X.copy(); original=data['TotalCharges']
    blanks=original.isna() | original.astype(str).str.strip().eq('')
    data['TotalCharges']=pd.to_numeric(original,errors='coerce')
    if data['TotalCharges'].isna().sum()!=blanks.sum():raise ValueError('Nonblank invalid TotalCharges')
    if not data.loc[blanks,'tenure'].eq(0).all():raise ValueError('Structural zero requires tenure zero')
    for col in NUMERIC:
        vals=pd.to_numeric(data[col],errors='raise')
        if (vals.dropna()<0).any() or not np.isfinite(vals.dropna()).all():raise ValueError('Invalid numeric value '+col)
        data[col]=vals
    return data

class TelcoSemanticTransformer(BaseEstimator, TransformerMixin):
    def __init__(self,native=False):self.native=native
    def fit(self,X,y=None):
        semantic_repair(X); self.n_features_in_=X.shape[1]
        return self
    def transform(self,X):
        data=semantic_repair(X)
        if self.native:
            data['TotalCharges']=data['TotalCharges'].fillna(0)
            if data.isna().any().any():raise ValueError('Unhandled CatBoost native missing value')
            for c in CATEGORICAL:data[c]=data[c].astype(str)
        return data

def preprocessing(scaling=True,totalcharges='zero'):
    if totalcharges not in ('zero','median'):raise ValueError(totalcharges)
    numeric_processor=Pipeline([('imputer',SimpleImputer(strategy='median'))] + ([('scaler',StandardScaler())] if scaling else []))
    charge_processor=Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value=0) if totalcharges=='zero' else SimpleImputer(strategy='median'))] + ([('scaler',StandardScaler())] if scaling else []))
    categoricals=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('nums',numeric_processor,['SeniorCitizen','tenure','MonthlyCharges']),
                              ('charges',charge_processor,['TotalCharges']),('categorical',categoricals,CATEGORICAL)])

def make_pipeline(estimator,scaling=False,balance='none',totalcharges='zero'):
    steps=[('semantic',TelcoSemanticTransformer()),('prepare',preprocessing(scaling,totalcharges))]
    if balance=='ros':steps.append(('sampler',RandomOverSampler(random_state=42)))
    est=clone(estimator)
    if balance=='weighted':est.set_params(class_weight='balanced')
    return ImbPipeline(steps+[('model',est)])

def make_cb():
    return CatBoostClassifier(iterations=500,depth=4,learning_rate=.03,loss_function='Logloss',
        random_seed=42,thread_count=2,verbose=False,allow_writing_files=False,
        cat_features=CATEGORICAL)

class NativeCatBoostClassifier(ClassifierMixin, BaseEstimator):
    # sklearn-compatible wrapper: CatBoost 1.2.8 lacks tags required by sklearn 1.8 Pipeline.
    # Fit still uses precisely the project's native categorical pathway.
    def __init__(self, iterations=500, depth=4, learning_rate=.03, random_seed=42):
        self.iterations=iterations;self.depth=depth;self.learning_rate=learning_rate;self.random_seed=random_seed
    def fit(self,X,y):
        self.prep_=TelcoSemanticTransformer(native=True).fit(X)
        prepared=self.prep_.transform(X)
        self.estimator_=CatBoostClassifier(iterations=self.iterations,depth=self.depth,
            learning_rate=self.learning_rate,loss_function='Logloss',random_seed=self.random_seed,
            thread_count=2,verbose=False,allow_writing_files=False)
        self.estimator_.fit(prepared,y,cat_features=CATEGORICAL)
        self.classes_=np.array([0,1]);return self
    def predict_proba(self,X):
        return self.estimator_.predict_proba(self.prep_.transform(X))
    def predict(self,X):
        return (self.predict_proba(X)[:,1]>=.5).astype(int)

def make_cat_pipeline():
    return NativeCatBoostClassifier()

membership=split.set_index('customerID').loc[raw.customerID]
assert membership.Split.value_counts().to_dict()=={'Development':5634,'FinalTest_LOCKED':1409}
fold_order=folds.sort_values('DevelopmentOrder').customerID
by_id=raw.set_index('customerID')
X_dev=by_id.loc[fold_order,RAW_FEATURES].reset_index(drop=True)
y_dev=by_id.loc[fold_order,'Churn'].eq('Yes').astype(int).reset_index(drop=True)
fold_idx=folds.set_index('customerID').loc[fold_order,'Fold'].to_numpy()
cv_splits=[(np.where(fold_idx!=i)[0],np.where(fold_idx==i)[0]) for i in range(5)]
assert len(X_dev)==5634 and set(fold_idx)==set(range(5))
print('Leakage-aware source:',X_dev.shape,'| five frozen folds:', [len(v) for _,v in cv_splits]);
display(X_dev.head(2))

Leakage-aware source: (5634, 19) | five frozen folds: [1127, 1127, 1127, 1127, 1126]


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Male,0,No,No,35,No,No phone service,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.2,1701.65
1,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,75.1,1151.55


### 2.1 Treatment and balancing decisions
The recorded experiments compared structural-zero vs median charging imputation, and no balancing vs class weights vs training-fold random oversampling. Mean-fold AP and recall can favor different decisions. Tables below are **recorded evidence from the submitted member notebook**.

In [4]:
display(source_csv('tc_treatment'))
display(source_csv('balance'))
print('Default: zero treatment only after tenure check; keep original class proportions because recorded AP comparison did not improve with balancing.')
# Prove safety of the transform on held-out fold, rather than reusing fitted preprocessing.
train_idx,valid_idx=cv_splits[0]
safe=make_pipeline(LogisticRegression(max_iter=2000,random_state=42),scaling=True)
safe.fit(X_dev.iloc[train_idx],y_dev.iloc[train_idx]); hold_proba=safe.predict_proba(X_dev.iloc[valid_idx])[:,1]
print('Successfully fit preprocessing inside fold 0; held-out AP:',round(average_precision_score(y_dev.iloc[valid_idx],hold_proba),4))

,Treatment,pr_auc,roc_auc,f1,precision,recall,delta_vs_best_pr_auc
0,MEDIAN,0.661430,0.846137,0.592316,0.652899,0.543144,-0.000226
1,MEDIAN+IND,0.661636,0.846203,0.592412,0.652094,0.543813,-0.000020
2,ZERO,0.661656,0.846201,0.592329,0.652927,0.543144,0.000000
3,ZERO+IND,0.661607,0.846219,0.592090,0.652304,0.543144,-0.000049


,Strategy,pr_auc,roc_auc,f1,precision,recall
0,BAL-0,0.661656,0.846201,0.592329,0.652927,0.543144
1,BAL-W,0.659985,0.845953,0.628257,0.516933,0.801338
2,BAL-ROS,0.658750,0.845744,0.627451,0.515870,0.801338


Default: zero treatment only after tenure check; keep original class proportions because recorded AP comparison did not improve with balancing.


Successfully fit preprocessing inside fold 0; held-out AP: 0.6439


### 2.2 Engineered telecom features (tested, not automatically retained)
`ServiceCount`, `SupportGap`, `AutoPay` and `ChargePerService` were tested by adding them to a linear baseline. They did **not** improve the recorded AP, so the final model uses the 19 raw features. Retaining a negative result is important evidence.

In [5]:
def engineered_features(df):
    x=semantic_repair(df)
    addons=['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    internet=x.InternetService.ne('No')
    count=x.PhoneService.eq('Yes').astype(int)+x.MultipleLines.eq('Yes').astype(int)+internet.astype(int)
    for col in addons:count=count+x[col].eq('Yes').astype(int)
    x['ServiceCount']=count
    x['SupportGap']=np.where(internet,x[['OnlineSecurity','DeviceProtection','TechSupport']].eq('No').sum(axis=1),0)
    x['AutoPay']=x.PaymentMethod.isin(['Bank transfer (automatic)','Credit card (automatic)']).astype(int)
    x['ChargePerService']=x.MonthlyCharges/x.ServiceCount.clip(lower=1)
    return x
print('Examples of deterministic engineered features:')
display(engineered_features(X_dev.head(3))[['ServiceCount','SupportGap','AutoPay','ChargePerService']])
display(source_csv('features'))
display(source_csv('demographic'))
print('The demographic-reduction comparison is an input-minimization experiment, not a fairness proof.')

Examples of deterministic engineered features:


,ServiceCount,SupportGap,AutoPay,ChargePerService
0,4,2,0,12.300000
1,3,2,0,25.033333
2,4,1,0,10.137500


,pr_auc,roc_auc,f1,precision,recall,FeatureSet,delta_pr_auc,Decision
0,0.661656,0.846201,0.592329,0.652927,0.543144,BASE,0.000000,REFERENCE
1,0.661580,0.846190,0.592114,0.652405,0.543144,BASE + ServiceCount,-0.000076,EXPERIMENTAL_NOT_DEFAULT
2,0.661533,0.846190,0.593054,0.653705,0.543813,BASE + SupportGap,-0.000123,EXPERIMENTAL_NOT_DEFAULT
3,0.661510,0.846137,0.591798,0.652611,0.542475,BASE + AutoPay,-0.000146,EXPERIMENTAL_NOT_DEFAULT
4,0.660973,0.845979,0.593718,0.652330,0.545819,BASE + ChargePerService,-0.000683,EXPERIMENTAL_NOT_DEFAULT
5,0.660977,0.845983,0.593538,0.651839,0.545819,BASE + ALL_ENGINEERED,-0.000679,EXPERIMENTAL_NOT_DEFAULT


,FeatureSet,pr_auc,roc_auc,f1,precision,recall,delta_pr_auc_vs_full
0,FS-RAW-FULL,0.661656,0.846201,0.592329,0.652927,0.543144,0.000000
1,FS-RAW-NOGENDER,0.660835,0.846145,0.594677,0.654691,0.545819,-0.000821
2,FS-RAW-LIGHT-DEMO,0.661672,0.846283,0.598212,0.657589,0.549833,0.000015
3,FS-RAW-NO-PROFILE,0.660117,0.846046,0.601222,0.655932,0.555853,-0.001539


The demographic-reduction comparison is an input-minimization experiment, not a fairness proof.


### 2.3 Customer-level preprocessing sensitivity
The original development-only OOF comparison held Logistic Regression and folds fixed while changing the imbalance policy. This is **not** the final 0.35 decision rule; the experiment used a 0.50 diagnostic threshold.

In [6]:
pre_p=source_csv('preprocess_pred').set_index('customerID').loc[fold_order].reset_index()
assert (pre_p.customerID.to_numpy()==fold_order.to_numpy()).all()
prep_unstable=pre_p['prep_any_disagreement'].astype(bool)
assert int(prep_unstable.sum())==1114
print(f'Development customers with disagreement across imbalance policies: {prep_unstable.sum()} / {len(pre_p)} ({prep_unstable.mean():.2%})')
display(pre_p[['customerID','pred_BAL-0','pred_BAL-W','pred_BAL-ROS','PrepStabilityFlag']].head(5))

Development customers with disagreement across imbalance policies: 1114 / 5634 (19.77%)


,customerID,pred_BAL-0,pred_BAL-W,pred_BAL-ROS,PrepStabilityFlag
0,4950-BDEUX,0,1,1,Sensitive
1,7993-NQLJE,0,1,1,Sensitive
2,7321-ZNSLA,0,0,0,Stable
3,4922-CVPDX,0,0,0,Stable
4,2903-YYTBW,0,1,1,Sensitive


## Member 2 decision and handoff
**Chosen default:** only set blank `TotalCharges` to zero where tenure is zero; otherwise reject invalid input. Fit category encoding and scaling on training folds. Keep original class proportions for the recorded primary AP comparison. Experimental engineered features remain non-default because the documented linear-model ablations did not improve AP.

The saved fold membership and `candidate_models_updated_oof.csv` allow Member 3 to compare model families on aligned development customers.

In [7]:
display(source_csv('prep_log').head(12))
display(source_csv('feature_log').head(8))
assert len(X_dev)==5634 and int(prep_unstable.sum())==1114

,ID,Issue,Options,Selected,Evidence,Status
0,PP-01,TotalCharges blanks,median | median+indicator | zero | zero+indicator,ZERO,All 11 blanks occur at tenure=0; CV PR-AUC dif...,APPROVED
1,PP-02,Structural service categories,preserve | collapse | convert missing,PRESERVE,No internet service / No phone service are val...,APPROVED
2,PP-03,Scaling,standard | none,MODEL_AWARE,Scale linear/distance representations; do not ...,APPROVED
3,PP-04,Class imbalance,none | class_weight | ROS,NO_SINGLE_WINNER,BAL-0 has strongest PR-AUC; BAL-W substantiall...,APPROVED_WITH_BRANCHES
4,PP-05,Categorical encoding,one-hot | native categorical | target encoding,MODEL_AWARE_OH_PLUS_NATIVE,One-hot for linear/sklearn tree pipeline; nati...,APPROVED
5,PP-06,Demographic feature policy,full | reduced,KEEP_FULL_REFERENCE_AND_LIGHT_DEMO_CHALLENGER,Removing gender + SeniorCitizen preserved base...,APPROVED
6,PP-07,Engineered telecom features,default | reject | experimental,EXPERIMENTAL_NOT_DEFAULT,Linear-model ablation showed no meaningful PR-...,APPROVED
7,PP-08,Preprocessing sensitivity data,final test | OOF development predictions,OOF_DEVELOPMENT_ONLY,Avoid consuming final test; each OOF customer ...,APPROVED
8,PP-09,Final test usage,development decisions | sealed,SEALED_FOR_MEMBER4,No preprocessing/model-selection predictions a...,APPROVED


,Feature,Origin,Type,AllowedUse,LeakageRisk,Status,FormulaOrSource,Notes
0,customerID,Raw,Identifier,Tracking/join only,ID_RISK,EXCLUDE,Raw dataset,Unique customer key; no behavioural meaning.
1,gender,Raw,Demographic/context,Predictor/audit candidate,Low,FULL_OR_MINIMIZED,Raw dataset,Included only if approved feature set uses it.
2,SeniorCitizen,Raw,Demographic/context,Predictor/audit candidate,Low,FULL_OR_MINIMIZED,Raw dataset,Stored as 0/1 but conceptually demographic cat...
3,Partner,Raw,Profile/context,Predictor candidate,Low,FULL_OR_MINIMIZED,Raw dataset,Association is not causal.
4,Dependents,Raw,Profile/context,Predictor candidate,Low,FULL_OR_MINIMIZED,Raw dataset,Association is not causal.
5,tenure,Raw,Relationship history,Predictor,Low,APPROVED,Raw dataset,Numeric customer history.
6,PhoneService,Raw,Service,Predictor,Low,APPROVED,Raw dataset,Structural upstream service state.
7,MultipleLines,Raw,Service,Predictor,Low,APPROVED,Raw dataset,Preserve No phone service.


## Member handoff and interpretation
This member notebook is a **stand-alone view** of one part of the shared project. It duplicates the minimum setup needed to run independently. The integrated notebook provides the complete sequential view across Members 1–4. Detailed decision logs and archived results remain in the `decision_logs/` and `model_method_comparison/` folders.

**Do not report saved evidence as a newly rerun search** if the notebook only loads the existing result CSV. Assessments of business benefit require new intervention-outcome data; model agreement is not a correctness guarantee.
